# Python reference 11: concurrency: threads, processes and asyncio

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/learning/python-reference-11-concurrency-and-async.ipynb)

The GIL and when to use threads, processes or `asyncio`; `concurrent.futures`; locks, queues and producer/consumer threads; and `asyncio` with `gather`, `to_thread`, semaphores, timeouts, `TaskGroup` and worker queues. Practise it in [Python practice 11: concurrency: threads, processes and asyncio](../practice/python-practice-11-concurrency-and-async.ipynb).

*Last reviewed: November 2026, against Python 3.13.*

## Contents

| section | entries |
| --- | --- |
| [Choosing a concurrency model](#choosing-a-concurrency-model) | [Threads, processes or asyncio?](#threads-processes-or-asyncio) |
| [concurrent.futures](#concurrentfutures) | [`ThreadPoolExecutor`](#threadpoolexecutor), [`ProcessPoolExecutor`](#processpoolexecutor) |
| [Threads and shared state](#threads-and-shared-state) | [Locks and race conditions](#locks-and-race-conditions), [Producer and consumer with `queue.Queue`](#producer-and-consumer-with-queuequeue) |
| [asyncio](#asyncio) | [Coroutines, `await` and `gather`](#coroutines-await-and-gather), [Limiting concurrency, timeouts and `TaskGroup`](#limiting-concurrency-timeouts-and-taskgroup), [Worker pools with `asyncio.Queue`](#worker-pools-with-asyncioqueue) |

## Choosing a concurrency model

### Threads, processes or asyncio?

CPython's **global interpreter lock (GIL)** lets one thread execute Python bytecode at a time. Threads still help
when they mostly **wait** (network, disk, database), because waiting releases the GIL; they do not speed up pure
Python computation.

| model | good for | cost / limits |
| --- | --- | --- |
| threads (`ThreadPoolExecutor`) | blocking I/O: HTTP calls, S3 reads, database queries with blocking drivers | shared memory needs locks; no CPU speed-up for Python code |
| processes (`ProcessPoolExecutor`) | CPU-bound Python: parsing, compression, pure-Python transforms | separate memory; arguments and results are pickled; start-up cost |
| `asyncio` | thousands of concurrent waits with async libraries (`aiohttp`, `asyncpg`) | everything in the call path must be async; one blocking call stalls the whole loop |

**Notes:** NumPy, pandas, Arrow and compression libraries release the GIL in their C code, so threads can speed them
up. Python 3.13 ships an optional free-threaded build (`python3.13t`) without the GIL; most deployments still use the
default build.

## concurrent.futures

### `ThreadPoolExecutor`

| call | returns |
| --- | --- |
| `with ThreadPoolExecutor(max_workers=8) as pool:` | a pool; leaving the block waits for submitted work |
| `pool.map(fn, items, timeout=None)` | results **in input order**; the first exception is raised when its result is reached |
| `pool.submit(fn, *args)` | a `Future` |
| `future.result(timeout=None)` | the value, or re-raises the call's exception |
| `as_completed(futures, timeout=None)` | futures in completion order |
| `wait(futures, return_when=FIRST_COMPLETED)` | `(done, not_done)` sets |

**Notes:** use `submit` + `as_completed` when you need per-item error handling or progress; `map` when all items
must succeed. Bound `max_workers` to what the remote system tolerates.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

def fetch(key):
    time.sleep(0.2)              # stands in for a network call
    if key == "bad":
        raise ConnectionError(f"cannot fetch {key}")
    return key.upper()

keys = ["a", "b", "c", "d", "e"]
start = time.perf_counter()
print([fetch(k) for k in keys], f"sequential {time.perf_counter() - start:.2f}s")
start = time.perf_counter()
with ThreadPoolExecutor(max_workers=5) as pool:
    print(list(pool.map(fetch, keys)), f"threads {time.perf_counter() - start:.2f}s")

with ThreadPoolExecutor(max_workers=4) as pool:
    futures = {pool.submit(fetch, k): k for k in ["x", "bad", "y"]}
    for future in as_completed(futures):
        key = futures[future]
        try:
            print("  ok", key, future.result())
        except ConnectionError as e:
            print("  failed", key, e)

### `ProcessPoolExecutor`

The same API, with work running in separate processes, so CPU-bound Python code uses several cores. Functions and
arguments must be picklable: define worker functions at module level (in a `.py` file), not as lambdas. Wrap the
pool in an `if __name__ == "__main__":` guard in scripts, because platforms that **spawn** processes re-import the
main module. `chunksize=` in `map` reduces overhead for many small items.

In [ ]:
from concurrent.futures import ProcessPoolExecutor

chunks = [range(i * 30_000_000, (i + 1) * 30_000_000) for i in range(4)]   # ranges pickle cheaply
start = time.perf_counter()
serial = [sum(c) for c in chunks]
serial_time = time.perf_counter() - start
start = time.perf_counter()
with ProcessPoolExecutor(max_workers=4) as pool:
    parallel = list(pool.map(sum, chunks))
print(parallel == serial, f"serial {serial_time:.2f}s, 4 processes {time.perf_counter() - start:.2f}s (including process start-up)")

## Threads and shared state

### Locks and race conditions

`counter += 1` is a read, an add and a write; two threads can interleave them and lose updates. Protect shared
mutable state with `threading.Lock` (`with lock:`), or avoid sharing: give each thread its own result and combine
results at the end (usually the better design).

| primitive | use |
| --- | --- |
| `Lock` / `RLock` | mutual exclusion (`RLock` can be re-acquired by the same thread) |
| `Event` | one thread signals others (`set()`, `wait(timeout)`) |
| `Semaphore(n)` | at most `n` threads in a section (connection limits) |
| `local()` | per-thread storage (a connection per thread) |

In [ ]:
import threading

def add_many(n, lock=None):
    global total
    for _ in range(n):
        if lock:
            with lock:
                total += 1
        else:
            value = total
            time.sleep(0)       # yield to another thread, making the race visible
            total = value + 1

for lock in (None, threading.Lock()):
    total = 0
    threads = [threading.Thread(target=add_many, args=(2_000, lock)) for _ in range(4)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    print("with lock" if lock else "no lock  ", total, "of", 8_000)

### Producer and consumer with `queue.Queue`

`queue.Queue` is a thread-safe FIFO: producers `put`, consumers `get`, and `maxsize` applies back-pressure so a fast
producer cannot exhaust memory. Stop consumers with a **sentinel** per consumer (or `queue.shutdown()` in 3.13), and
call `task_done()` / `join()` to wait for processing.

In [ ]:
import queue

work, results, STOP = queue.Queue(maxsize=10), queue.Queue(), object()

def consumer():
    while (item := work.get()) is not STOP:
        results.put(item * item)
        work.task_done()
    work.task_done()

workers = [threading.Thread(target=consumer) for _ in range(3)]
for w in workers:
    w.start()
for i in range(20):
    work.put(i)
for _ in workers:
    work.put(STOP)
work.join()
print(sum(results.get() for _ in range(results.qsize())))

## asyncio

### Coroutines, `await` and `gather`

`async def` defines a coroutine function; calling it creates a coroutine object that does nothing until awaited.
`await` suspends the current coroutine until the awaited one finishes, letting the event loop run others meanwhile.

| call | does |
| --- | --- |
| `asyncio.run(main())` | start an event loop and run `main()` (scripts; not inside Jupyter, which already runs a loop: use top-level `await`) |
| `await asyncio.gather(*coros, return_exceptions=False)` | run concurrently; results in argument order |
| `asyncio.create_task(coro)` | schedule a coroutine to run in the background; keep a reference to the task |
| `await asyncio.sleep(s)` | non-blocking sleep (never `time.sleep` inside async code) |
| `await asyncio.to_thread(blocking_fn, *args)` | run blocking code in a thread without stalling the loop |

**Notes:** the cells below use top-level `await`, which works in Jupyter and IPython. In a script, wrap the code in
`async def main()` and call `asyncio.run(main())`.

In [ ]:
import asyncio

async def fetch_async(key, delay=0.2):
    await asyncio.sleep(delay)
    return key.upper()

start = time.perf_counter()
results = await asyncio.gather(*(fetch_async(k) for k in "abcdefghij"))
print(results, f"{time.perf_counter() - start:.2f}s for 10 waits of 0.2s")

def blocking_read():
    time.sleep(0.2)
    return "file contents"

start = time.perf_counter()
print(await asyncio.gather(asyncio.to_thread(blocking_read), fetch_async("x")), f"{time.perf_counter() - start:.2f}s")

### Limiting concurrency, timeouts and `TaskGroup`

| tool | use |
| --- | --- |
| `asyncio.Semaphore(n)` | at most `n` coroutines inside `async with sem:` (API rate limits, connection pools) |
| `async with asyncio.timeout(s):` (3.11+) / `asyncio.wait_for(coro, s)` | cancel work that takes too long (`TimeoutError`) |
| `async with asyncio.TaskGroup() as tg:` (3.11+) | structured concurrency: `tg.create_task(...)`; if one task fails the rest are cancelled and errors are raised as an `ExceptionGroup` |
| `asyncio.Queue` | producer and worker coroutines, like `queue.Queue` for threads |

In [ ]:
sem = asyncio.Semaphore(3)
active = peak = 0

async def limited(i):
    global active, peak
    async with sem:
        active += 1
        peak = max(peak, active)
        await asyncio.sleep(0.05)
        active -= 1
        return i

print(await asyncio.gather(*(limited(i) for i in range(10))), "peak concurrency:", peak)

try:
    async with asyncio.timeout(0.1):
        await asyncio.sleep(1)
except TimeoutError:
    print("timed out after 0.1s")

async def maybe_fail(i):
    await asyncio.sleep(0.01 * i)
    if i == 2:
        raise ValueError(f"task {i} failed")
    return i

try:
    async with asyncio.TaskGroup() as tg:
        tasks = [tg.create_task(maybe_fail(i)) for i in range(5)]
except* ValueError as group:
    print("TaskGroup:", [str(e) for e in group.exceptions], "| cancelled:", sum(t.cancelled() for t in tasks))

### Worker pools with `asyncio.Queue`

A fixed number of worker coroutines pulling from a queue gives bounded concurrency over a stream of work and lets
you stop cleanly: `await queue.join()` waits until every item is processed, then cancel the workers.

In [ ]:
async def worker(name, q, out):
    while True:
        item = await q.get()
        try:
            await asyncio.sleep(0.01)
            out.append((name, item * 10))
        finally:
            q.task_done()

q, out = asyncio.Queue(), []
for i in range(12):
    q.put_nowait(i)
workers = [asyncio.create_task(worker(f"w{n}", q, out)) for n in range(3)]
await q.join()
for w in workers:
    w.cancel()
await asyncio.gather(*workers, return_exceptions=True)
print(sorted(v for _, v in out), "| items per worker:", {w: sum(1 for n, _ in out if n == w) for w in ("w0", "w1", "w2")})

**Next:** practise these entries in [Python practice 11: concurrency: threads, processes and asyncio](../practice/python-practice-11-concurrency-and-async.ipynb).